# Coding agent chấm bằng test pass/fail (portfolio)

Agent nhận một yêu cầu (`TASK.md`) và một repo nhỏ, phải sửa hoặc viết mã cho tới khi test pass. Kết quả **không**
chấm bằng mắt hay bằng LLM: mã cuối cùng của agent được chép sang một thư mục sạch cùng **test ẩn** (agent không thấy)
và chạy pytest trong sandbox (`sandbox.py`). Bộ 12 task ghim ở `tasks/`.

**Cách làm bài**
- `sandbox.py`: ba hàm TODO, có test (`uv run pytest tracks/ai-engineer/modules/16-ai-agents/exercises/advanced/01-coding-agent -q`).
- Notebook này: thiết kế agent của bạn (tool, prompt, vòng lặp), chạy trên cả bộ task, đo pass@1, phân loại lỗi.
  Không có test cho phần notebook; chấm theo rubric trong `SOLUTION.md`.
- Viết `REPORT.md` theo khung có sẵn.

In [ ]:
import os

os.environ.setdefault("OMP_NUM_THREADS", "4")

import json
import re
import tempfile
import time
from dataclasses import dataclass, field
from pathlib import Path

import pandas as pd

from sandbox import TestRun, grade, run_pytest, safe_path, source_files

In [ ]:
# explore: bộ task và một task mẫu
TASKS = Path("tasks")
task_dirs = sorted(p for p in TASKS.iterdir() if p.is_dir() and p.name[:2].isdigit())
print([p.name for p in task_dirs])
sample = task_dirs[2]
print((sample / "TASK.md").read_text(encoding="utf-8"))
for p in sorted((sample / "repo").rglob("*.py")):
    print(f"--- {p.relative_to(sample / 'repo')} ---\n{p.read_text(encoding='utf-8')}")

## 1. Môi trường của agent

Agent làm việc trong một bản sao của `task/repo` ở thư mục tạm. Tối thiểu cần các tool: liệt kê tệp, đọc tệp, ghi tệp,
chạy test hiện có (`run_pytest` của bạn), và một cách kết thúc. Mọi đường dẫn đi qua `safe_path`. Ghi lại những gì
agent làm để phân tích lỗi sau: số lần ghi, số lần chạy test, có sửa tệp trong `tests/` không.

Quyết định thiết kế giao diện agent-máy (agent-computer interface, mục 7 lý thuyết) là phần quan trọng nhất của bài:
ghi tệp bằng JSON (`{"path": ..., "content": "..."}`, model phải escape mọi xuống dòng và dấu nháy) hay bằng khối code
sau `Action Input`; đưa sẵn nội dung tệp vào prompt đầu hay để agent tự đọc; output của test dài bao nhiêu; xử lý
action lặp thế nào.

In [ ]:
class CodingEnv:
    """Repo làm việc của agent và các tool của nó. Gợi ý giao diện: specs() và call(name, args, reply) như registry."""

    def __init__(self, task_dir: Path, workdir: Path, test_timeout: float = 20):
        # TODO
        raise NotImplementedError

## 2. Vòng lặp agent

Dùng lại `parse_react` / `run_react` của bài Basic 02 (dán vào đây) hoặc viết vòng riêng nếu giao diện ghi tệp cần
đọc thêm phần sau `Action Input`. Trả về một bản ghi đủ để chấm và phân tích: task, pass/fail theo test ẩn, pass/fail
theo test thấy được, lý do dừng, số bước, số lần ghi, số lần chạy test, số lỗi định dạng, có sửa test không, thời gian,
trace.

`llmkit.chat` không gửi `seed` và `num_ctx`. Coding agent hay vượt context mặc định 4 096 token của Ollama (mã + output
test), nên client dưới đây gọi thẳng `/api/chat` với `num_ctx=8192` và `seed` cố định.

In [ ]:
# explore: client Ollama có seed và num_ctx
import httpx

import llmkit


class OllamaChat:
    def __init__(self, model="qwen2.5:3b", seed=0, temperature=0.0, num_ctx=8192):
        self.model, self.options = model, {"temperature": temperature, "seed": seed, "num_ctx": num_ctx}
        self.http = httpx.Client(timeout=600)

    def chat(self, messages, *, model=None, temperature=None, json_schema=None):
        payload = {"model": self.model, "messages": messages, "stream": False, "options": self.options}
        if json_schema is not None:
            payload["format"] = json_schema
        r = self.http.post(os.environ.get("OLLAMA_HOST", "http://localhost:11434") + "/api/chat", json=payload)
        if r.status_code >= 400:
            raise llmkit.LLMError(f"HTTP {r.status_code}: {r.text[:200]}")
        return r.json()["message"]["content"]

In [ ]:
def run_coding_agent(llm, task_dir: Path, max_steps: int = 12) -> dict:
    # TODO
    raise NotImplementedError

## 3. Chạy cả bộ task, pass@1, phân loại lỗi

pass@1 với `temperature` 0 là một lần chạy mỗi task. Để biết con số dao động bao nhiêu, chạy thêm vài seed với
`temperature` 0.7 và báo khoảng (mục 7 lý thuyết: pass@k và cách ước lượng không chệch). Phân loại mỗi task fail theo
một taxonomy bạn tự định nghĩa từ trace (ví dụ: không bao giờ ghi tệp, lỗi định dạng, hết bước, test thấy được pass
nhưng test ẩn fail, sửa test thay vì sửa mã...). Cell dưới để bạn điền.

In [ ]:
# explore: chạy cả bộ (TODO của bạn: gọi run_coding_agent cho mọi task, lưu kết quả ra JSONL, in bảng)